In [ ]:
"""
Batch TF-IDF Embedding Interpretation
=====================================

This script extracts and generalizes the TF-IDF interpretation
pipeline from the v9 Tampa Active Travel analysis.

It automatically detects and processes ALL:
    zone_embeddings*.csv

Examples:
    zone_embeddings.csv
    zone_embeddings_minilm.csv
    zone_embeddings_mpnet.csv
    zone_embeddings_openai.csv
    zone_embeddings_e5.csv

For each embedding backend, the script:
    1. Loads embedding dimensions
    2. Aligns zoning corpora
    3. Builds TF-IDF features
    4. Correlates embedding dimensions with terms
    5. Filters zoning abbreviations + legal boilerplate
    6. Produces:
         - CSV interpretation table
         - formatted XLSX workbook

Outputs:
    zone_embeddings*_tfidf_interpretation.csv
    zone_embeddings*_tfidf_interpretation.xlsx

Required:
    zone_text_corpora.csv

Required embedding columns:
    zone_class
    emb_000, emb_001, ...

Required corpus columns:
    zone_class
    corpus
"""

import os
import glob
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ─────────────────────────────────────────────────────────────
# SETTINGS
# ─────────────────────────────────────────────────────────────

CORPUS_PATH = "zone_text_corpora.csv"

MAX_FEATURES = 1000
NGRAM_RANGE = (1, 2)
MIN_DF = 2

# Top 6 positive + 4 negative terms per dimension: enough to characterize
# each pole without overwhelming the reader. Ranked by absolute correlation.
TOP_POS_TERMS = 6
TOP_NEG_TERMS = 4

# ─────────────────────────────────────────────────────────────
# FILTER TERMS
# ─────────────────────────────────────────────────────────────

_ZONING_STOPWORDS = {
    'rs', 'rm', 'ro', 'op', 'cn', 'cg', 'ci', 'ig', 'ih', 'pd', 'cbd',
    'nmu', 'yc', 'cd', 'cu', 'uc', 'sh', 'ap', 'map', 'bti',
    'sec', 'shall', 'section', 'chapter', 'construed', 'pb',
    'pursuant', 'thereof', 'herein', 'whereas', 'notwithstanding',
}

# ─────────────────────────────────────────────────────────────
# FIND EMBEDDING FILES
# ─────────────────────────────────────────────────────────────

embedding_files = sorted(glob.glob("zone_embeddings*.csv"))

if len(embedding_files) == 0:
    raise ValueError("No zone_embeddings*.csv files found.")

print("\nDetected embedding files:")
for f in embedding_files:
    print(f"  - {f}")

# ─────────────────────────────────────────────────────────────
# LOAD CORPORA
# ─────────────────────────────────────────────────────────────

print("\nLoading zoning corpora...")
zone_corp = pd.read_csv(CORPUS_PATH)

if "zone_class" not in zone_corp.columns:
    raise ValueError("zone_text_corpora.csv missing 'zone_class'")

if "corpus" not in zone_corp.columns:
    raise ValueError("zone_text_corpora.csv missing 'corpus'")

# ─────────────────────────────────────────────────────────────
# PROCESS EACH EMBEDDING FILE
# ─────────────────────────────────────────────────────────────

for EMB_PATH in embedding_files:

    print("\n" + "█" * 72)
    print(f"RUNNING: {EMB_PATH}")
    print("█" * 72)

    # ---------------------------------------------------------
    # LOAD EMBEDDINGS
    # ---------------------------------------------------------

    zone_emb = pd.read_csv(EMB_PATH)

    if "zone_class" not in zone_emb.columns:
        print("Skipping: missing zone_class column.")
        continue

    emb_cols = [c for c in zone_emb.columns if c.startswith("emb_")]

    if len(emb_cols) == 0:
        print("Skipping: no embedding columns.")
        continue

    print(f"\nZones: {len(zone_emb)}")
    print(f"Embedding dimensions: {len(emb_cols)}")

    # ---------------------------------------------------------
    # ALIGN CORPORA
    # ---------------------------------------------------------

    all_zones = zone_emb["zone_class"].tolist()

    corp_aligned = (
        zone_corp
        .set_index("zone_class")
        .reindex(all_zones)["corpus"]
        .fillna("")
    )

    # ---------------------------------------------------------
    # TF-IDF
    # ---------------------------------------------------------

    print("\nBuilding TF-IDF matrix...")

    # Correlate embedding dims with TF-IDF terms to give human-readable labels
    # to opaque PCA dimensions. Pearson r(emb_dim, tfidf_term) across 56 zones
    # reveals which regulatory concepts each dimension captures.
    tfidf_interp = TfidfVectorizer(
        max_features=MAX_FEATURES,
        ngram_range=NGRAM_RANGE,
        stop_words='english',
        min_df=MIN_DF,
        sublinear_tf=True,

        # 3+ letter alphabetic words only -- excludes numeric tokens and
        # zone codes (RS-60 etc.) that would dominate correlations.
        token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]{2,}\b',
    )

    tfidf_matrix = tfidf_interp.fit_transform(
        corp_aligned
    ).toarray()

    terms = tfidf_interp.get_feature_names_out()

    print(f"TF-IDF terms retained: {len(terms)}")

    # ---------------------------------------------------------
    # TERM FILTERING
    # ---------------------------------------------------------

    term_mask = np.array([
        not any(tok in _ZONING_STOPWORDS
                for tok in t.lower().split())
        for t in terms
    ])

    print(
        f"Filtered out {(~term_mask).sum()} zoning/legalese terms"
    )

    # ---------------------------------------------------------
    # CORRELATIONS
    # ---------------------------------------------------------

    print("\nComputing embedding-term correlations...")

    emb_matrix = zone_emb[emb_cols].values

    n_emb = len(emb_cols)

    all_mat = np.hstack([emb_matrix, tfidf_matrix])

    corr_full = np.corrcoef(all_mat.T)

    emb_term_corr = corr_full[:n_emb, n_emb:]

    # Zero-out filtered terms
    emb_term_corr[:, ~term_mask] = 0.0

    # ---------------------------------------------------------
    # BUILD INTERPRETATION TABLE
    # ---------------------------------------------------------

    print("\nGenerating interpretation table...")

    rows_out = []

    for dim_idx, col_name in enumerate(emb_cols):

        corrs = emb_term_corr[dim_idx]

        # Positive terms
        top_pos_idx = np.argsort(corrs)[-TOP_POS_TERMS:][::-1]

        # Negative terms
        top_neg_idx = np.argsort(corrs)[:TOP_NEG_TERMS]

        pos_terms = [terms[i] for i in top_pos_idx]
        pos_corrs = [float(corrs[i]) for i in top_pos_idx]

        neg_terms = [terms[i] for i in top_neg_idx]
        neg_corrs = [float(corrs[i]) for i in top_neg_idx]

        # Label
        label = " / ".join(pos_terms[:2])

        # Embedding stats
        dim_values = emb_matrix[:, dim_idx]

        dim_mean = float(np.mean(dim_values))
        dim_std = float(np.std(dim_values))
        dim_min = float(np.min(dim_values))
        dim_max = float(np.max(dim_values))
        dim_range = dim_max - dim_min

        max_abs_corr = float(np.max(np.abs(corrs)))

        # Highest/lowest zones
        zone_highest = all_zones[int(np.argmax(dim_values))]
        zone_lowest = all_zones[int(np.argmin(dim_values))]

        row_dict = {
            'dimension': col_name,
            'label': label,

            'emb_mean': dim_mean,
            'emb_std': dim_std,
            'emb_min': dim_min,
            'emb_max': dim_max,
            'emb_range': dim_range,

            'zone_highest': zone_highest,
            'zone_lowest': zone_lowest,

            'max_abs_term_corr': max_abs_corr,
        }

        for k in range(TOP_POS_TERMS):
            row_dict[f'pos_term_{k+1}'] = pos_terms[k]
            row_dict[f'pos_corr_{k+1}'] = pos_corrs[k]

        for k in range(TOP_NEG_TERMS):
            row_dict[f'neg_term_{k+1}'] = neg_terms[k]
            row_dict[f'neg_corr_{k+1}'] = neg_corrs[k]

        rows_out.append(row_dict)

    # ---------------------------------------------------------
    # CREATE DATAFRAME
    # ---------------------------------------------------------

    emb_table = pd.DataFrame(rows_out)

    emb_table = emb_table.sort_values(
        'max_abs_term_corr',
        ascending=False
    ).reset_index(drop=True)

    emb_table.insert(
        0,
        'rank',
        range(1, len(emb_table) + 1)
    )

    # ---------------------------------------------------------
    # OUTPUT PATHS
    # ---------------------------------------------------------

    base_name = os.path.splitext(
        os.path.basename(EMB_PATH)
    )[0]

    CSV_PATH = f"{base_name}_tfidf_interpretation.csv"
    XLSX_PATH = f"{base_name}_tfidf_interpretation.xlsx"

    # ---------------------------------------------------------
    # SAVE CSV
    # ---------------------------------------------------------

    emb_table.to_csv(CSV_PATH, index=False)

    print(f"\nSaved CSV: {CSV_PATH}")

    # ---------------------------------------------------------
    # BUILD FORMATTED XLSX
    # ---------------------------------------------------------

    wb = Workbook()
    ws = wb.active
    ws.title = "Embedding Dimensions"

    col_spec = [
        ('Rank',               5,  '0'),
        ('Dimension',         11,  None),
        ('Interpretive Label',30,  None),
        ('Mean',               8,  '0.000'),
        ('Std',                8,  '0.000'),
        ('Min',                8,  '0.000'),
        ('Max',                8,  '0.000'),
        ('Range',              8,  '0.000'),
        ('Zone\n(Highest)',   12,  None),
        ('Zone\n(Lowest)',    12,  None),
        ('Max |r|',            8,  '0.000'),

        ('Pos Term 1',        16,  None), ('r₁', 6, '+0.000'),
        ('Pos Term 2',        16,  None), ('r₂', 6, '+0.000'),
        ('Pos Term 3',        16,  None), ('r₃', 6, '+0.000'),
        ('Pos Term 4',        16,  None), ('r₄', 6, '+0.000'),
        ('Pos Term 5',        16,  None), ('r₅', 6, '+0.000'),
        ('Pos Term 6',        16,  None), ('r₆', 6, '+0.000'),

        ('Neg Term 1',        16,  None), ('r₁', 6, '+0.000'),
        ('Neg Term 2',        16,  None), ('r₂', 6, '+0.000'),
        ('Neg Term 3',        16,  None), ('r₃', 6, '+0.000'),
        ('Neg Term 4',        16,  None), ('r₄', 6, '+0.000'),
    ]

    data_keys = [
        'rank',
        'dimension',
        'label',

        'emb_mean',
        'emb_std',
        'emb_min',
        'emb_max',
        'emb_range',

        'zone_highest',
        'zone_lowest',

        'max_abs_term_corr',

        'pos_term_1', 'pos_corr_1',
        'pos_term_2', 'pos_corr_2',
        'pos_term_3', 'pos_corr_3',
        'pos_term_4', 'pos_corr_4',
        'pos_term_5', 'pos_corr_5',
        'pos_term_6', 'pos_corr_6',

        'neg_term_1', 'neg_corr_1',
        'neg_term_2', 'neg_corr_2',
        'neg_term_3', 'neg_corr_3',
        'neg_term_4', 'neg_corr_4',
    ]

    hdr_font = Font(
        bold=True,
        size=10,
        name='Arial',
        color='FFFFFF'
    )

    hdr_fill = PatternFill(
        'solid',
        fgColor='2F5496'
    )

    body_font = Font(
        size=9,
        name='Arial'
    )

    thin_bdr = Border(
        bottom=Side(
            style='thin',
            color='D0D0D0'
        )
    )

    # Headers
    for c, (hdr, width, _) in enumerate(col_spec, 1):

        cell = ws.cell(
            row=1,
            column=c,
            value=hdr
        )

        cell.font = hdr_font
        cell.fill = hdr_fill

        cell.alignment = Alignment(
            horizontal='center',
            vertical='center',
            wrap_text=True
        )

        ws.column_dimensions[
            get_column_letter(c)
        ].width = width

    ws.row_dimensions[1].height = 32

    # Body
    for r_idx, (_, row) in enumerate(emb_table.iterrows()):

        r = r_idx + 2

        for c, key in enumerate(data_keys, 1):

            val = row[key]

            cell = ws.cell(
                row=r,
                column=c,
                value=val
            )

            _, _, num_fmt = col_spec[c - 1]

            cell.font = body_font
            cell.border = thin_bdr

            cell.alignment = Alignment(
                vertical='center'
            )

            if (
                num_fmt and
                isinstance(val, (
                    int,
                    float,
                    np.integer,
                    np.floating
                ))
            ):
                cell.number_format = num_fmt

            # Color correlations
            if (
                'corr' in key and
                isinstance(val, (
                    float,
                    np.floating
                ))
            ):
                color = (
                    '006100'
                    if val > 0
                    else '9C0006'
                )

                cell.font = Font(
                    size=9,
                    name='Arial',
                    color=color
                )

            # Color term text
            if 'pos_term' in key and isinstance(val, str):
                cell.font = Font(
                    size=9,
                    name='Arial',
                    color='006100'
                )

            if 'neg_term' in key and isinstance(val, str):
                cell.font = Font(
                    size=9,
                    name='Arial',
                    color='9C0006'
                )

    ws.auto_filter.ref = (
        f"A1:{get_column_letter(len(col_spec))}"
        f"{len(emb_table)+1}"
    )

    ws.freeze_panes = 'D2'

    # Flat sheet
    ws2 = wb.create_sheet('Flat')

    for c, col in enumerate(emb_table.columns, 1):

        ws2.cell(
            row=1,
            column=c,
            value=col
        ).font = Font(
            bold=True,
            size=10
        )

    for r_idx, (_, row) in enumerate(emb_table.iterrows()):

        for c, col in enumerate(emb_table.columns, 1):

            ws2.cell(
                row=r_idx + 2,
                column=c,
                value=row[col]
            )

    wb.save(XLSX_PATH)

    print(f"Saved XLSX: {XLSX_PATH}")
    # ─────────────────────────────────────────────────────────────
    # CLEAN TERM SUMMARY TABLE (NO NUMERIC METRICS)
    # ─────────────────────────────────────────────────────────────

    clean_rows = []

    for _, row in emb_table.iterrows():

        pos_terms = [
            row['pos_term_1'],
            row['pos_term_2'],
            row['pos_term_3'],
            row['pos_term_4'],
            row['pos_term_5'],
            row['pos_term_6'],
        ]

        neg_terms = [
            row['neg_term_1'],
            row['neg_term_2'],
            row['neg_term_3'],
            row['neg_term_4'],
        ]

        # Remove duplicates while preserving order
        pos_terms = list(dict.fromkeys([t for t in pos_terms if pd.notna(t)]))
        neg_terms = list(dict.fromkeys([t for t in neg_terms if pd.notna(t)]))

        clean_rows.append({
            'rank': row['rank'],
            'dimension': row['dimension'],
            'label': row['label'],
            'positive_terms': ', '.join(pos_terms),
            'negative_terms': ', '.join(neg_terms),
        })

    clean_summary = pd.DataFrame(clean_rows)

    # Sort by importance rank
    clean_summary = clean_summary.sort_values('rank')

    # Save outputs
    clean_summary.to_csv(
        'embedding_dimension_term_summary.csv',
        index=False
    )

    clean_summary.to_excel(
        'embedding_dimension_term_summary.xlsx',
        index=False
    )

    print("\nSaved: embedding_dimension_term_summary.csv")
    print("Saved: embedding_dimension_term_summary.xlsx")

    # Console preview
    print("\n── Clean Embedding Term Summary ──")
    print(clean_summary.head(20).to_string(index=False))
    # ---------------------------------------------------------
    # CONSOLE SUMMARY
    # ---------------------------------------------------------

    print("\nTop interpretable dimensions:")

    for _, row in emb_table.head(15).iterrows():

        print(
            f"  {row['dimension']:<10}"
            f" | max|r|={row['max_abs_term_corr']:.3f}"
            f" | {row['label']}"
        )

print("\n✓ Batch TF-IDF interpretation complete.")

In [ ]:
#the final file names are df (origin and destination in the county area),destination county, and origin county. df is the only one with geometry